# esc50

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

repo_dir = Path("/kaggle/working/eX-LAViTs")
if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/VuTrinhNguyenHoang/eX-LAViTs.git", str(repo_dir)],
        check=True,
    )
os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))
print("Code:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


In [ ]:
%pip install -q timm==1.0.30 scikit-learn psutil soundfile

In [ ]:
import numpy as np
import pandas as pd
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from IPython.display import display
from src.models.linear_vit import create_linear_attention_vit
from src.utils.metrics import compute_loss_and_accuracy
from src.utils.revision_evaluation import (
    evaluate_methods,
    format_summary,
    save_metadata,
    benchmark_methods,
)


## Configuration

In [ ]:
DATASET = "esc50"
model_dir = Path("path/to/dir")
data_dir = Path("/kaggle/input/datasets/trnhvu/esc-50/ESC-50-master")
output_dir = Path("/kaggle/working/results") / DATASET
output_dir.mkdir(parents=True, exist_ok=True)
IMAGE_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 4
MAX_SAMPLES = None
STEPS = 20
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(SEED)
torch.manual_seed(SEED)
print(
    "Device:", DEVICE, "GPU:", torch.cuda.get_device_name() if DEVICE.type == "cuda" else None
)
print("Output:", output_dir)


## Data

In [ ]:
from sklearn.model_selection import train_test_split

esc_df = pd.read_csv(data_dir / "meta" / "esc50.csv")
esc_df["path"] = esc_df["filename"].map(lambda value: str(data_dir / "audio" / value))
esc_df["label_idx"] = esc_df["target"].astype(int)
idx2label = dict(esc_df.drop_duplicates("label_idx")[["label_idx", "category"]].values)
CLASSES = [idx2label[index] for index in range(len(idx2label))]
test_df = esc_df[esc_df["fold"] == 1].copy().reset_index(drop=True)
trainval_df = esc_df[esc_df["fold"] != 1].copy().reset_index(drop=True)
train_df, valid_df = train_test_split(
    trainval_df, test_size=0.1, random_state=SEED, stratify=trainval_df["label_idx"]
)

import torchaudio
import soundfile as sf
import torchvision.transforms as T
from src.utils.speech_preprocessing import Speech2Image, Speech2ImageConfig

IMAGENET_NORM = T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))


class ESC50Dataset(Dataset):

    def __init__(self, df, s2i_cfg=None, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.s2i = Speech2Image(
            s2i_cfg
            or Speech2ImageConfig(
                sample_rate=16000,
                window_sec=5.0,
                n_fft=512,
                win_length=400,
                hop_length=160,
                n_mels=224,
                fmin=20.0,
                fmax=7600.0,
                out_hw=224,
            )
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        waveform, sr = sf.read(row["path"], dtype="float32", always_2d=True)
        wav = torch.from_numpy(waveform.T.copy())
        if wav.size(0) > 1:
            wav = wav.mean(dim=0)
        else:
            wav = wav.squeeze(0)
        if sr != 16000:
            wav = torchaudio.functional.resample(wav, sr, 16000)
        x = self.s2i(wav)
        if self.transform is not None:
            x = self.transform(x)
        y = torch.tensor(int(row["label_idx"])).long()
        return (x, y)


test_dataset = ESC50Dataset(test_df, transform=IMAGENET_NORM)
sample_ids = test_dataset.df["path"].tolist()
class_names = (
    [label for label, index in sorted(label2idx.items(), key=lambda pair: pair[1])]
    if DATASET == "speechcommands"
    else CLASSES
)
n_classes = len(class_names)
test_dataset.df.to_csv(output_dir / "test_manifest.csv", index=False)
print("Train / validation / test:", len(train_df), len(valid_df), len(test_dataset))


## Model

In [ ]:
checkpoint_path = model_dir / f"{DATASET}.pth"
model = create_linear_attention_vit(
    "vit_tiny_patch16_224",
    pretrained=False,
    num_classes=n_classes,
    img_size=IMAGE_SIZE,
    verbose=False,
)
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
state = checkpoint["model_state_dict"] if "model_state_dict" in checkpoint else checkpoint
model.load_state_dict(state, strict=True)
del checkpoint, state
model.to(DEVICE).eval()
print("Parameters:", sum(parameter.numel() for parameter in model.parameters()))


In [ ]:
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == "cuda",
)
classification = compute_loss_and_accuracy(
    model, test_loader, nn.CrossEntropyLoss(), DEVICE, show_progress=True
)
print(classification)
(output_dir / "classification.json").write_text(json.dumps(classification, indent=2))


## Evaluation

In [ ]:
config = dict(
    dataset=DATASET,
    seed=SEED,
    image_size=IMAGE_SIZE,
    max_samples=MAX_SAMPLES,
    steps=STEPS,
    split_protocol="original ESC50 test fold 1; validation stratified 10% of folds 2-5; single test fold",
    class_names=class_names,
    data_dir=str(data_dir),
    model_name="vit_tiny_patch16_224",
    kernel="elu",
    eps=1e-6,
    normalization_mean=[0.485, 0.456, 0.406],
    normalization_std=[0.229, 0.224, 0.225],
)
config["audio_frontend"] = vars(test_dataset.s2i.cfg)
config["audio_decoder"] = "soundfile float32; mono channel mean; resample with torchaudio"
metadata = save_metadata(output_dir, config, model, checkpoint_path)


In [ ]:
summary = evaluate_methods(
    model,
    test_dataset,
    DEVICE,
    output_dir,
    max_samples=MAX_SAMPLES,
    steps=STEPS,
    sample_ids=sample_ids,
)
display(format_summary(summary))


## Compute

In [ ]:
xb = test_dataset[0][0].unsqueeze(0).to(DEVICE)
compute = benchmark_methods(model, xb, output_dir)
display(compute)
print("Saved:", output_dir)
